# AuraFit Platform: AI Capabilities Showcase
This notebook demonstrates the core AI functionalities of the AuraFit platform:
1. **Virtual Try-On Engine**
2. **Body Measurement Estimation**
3. **Product Recommendation Engine**

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import cv2

# Ensure the backend directory is in the path so we can import our modules
sys.path.append(os.getcwd())

from tryon import process_tryon
from classifier import classify_garment
from measurement import estimate_measurements
from recommender import get_recommendations

def display_img(path, title=""):
    img = cv2.imread(path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img)
        plt.title(title)
        plt.axis('off')
        plt.show()
    else:
        print(f"Image not found: {path}")


## Task 1: Virtual Garment Try-On
Here we take a user's reference photo and a garment photo, classify the garment type, and overlay it accurately on the user's body.

In [ ]:
person_img_path = 'uploads/demo_person.jpg'
garment_img_path = 'uploads/demo_garment.jpg'
output_img_path = 'outputs/demo_tryon_result.jpg'

print("--- INPUTS ---")
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
img1 = cv2.cvtColor(cv2.imread(person_img_path), cv2.COLOR_BGR2RGB)
plt.imshow(img1); plt.title('Person (Profile)'); plt.axis('off')
plt.subplot(1, 2, 2)
img2 = cv2.cvtColor(cv2.imread(garment_img_path), cv2.COLOR_BGR2RGB)
plt.imshow(img2); plt.title('Garment'); plt.axis('off')
plt.show()

print("Classifying garment...")
category_result = classify_garment(garment_img_path)
category = category_result.get('category', 'Top')
print(f"Detected Category: {category} ({category_result.get('confidence')} confidence)")

print("\nRunning Virtual Try-On... (this applies perspective warping & alpha blending)")
success = process_tryon(person_img_path, garment_img_path, output_img_path, category)

if success:
    print("\n--- OUTPUT ---")
    display_img(output_img_path, "AI Try-On Result")
else:
    print("Try-On Failed.")

## Task 2: Body Measurement Estimation
Using Mediapipe Pose Landmarks and the user's height, we estimate their body dimensions to recommend the perfect size.

In [ ]:
user_height_cm = 170

print(f"Estimating measurements for user with height {user_height_cm} cm...")
measurements = estimate_measurements(person_img_path, user_height_cm)

import pandas as pd
from IPython.display import display

df_meas = pd.DataFrame(list(measurements.items()), columns=['Metric', 'Value'])
display(df_meas.style.hide(axis='index'))

## Task 3: Product Recommendation Engine
A content-based recommender that filters by the user's community storefront, then ranks by category match, tag similarity, rating, and budget.

In [ ]:
user_profile = {
    "community": "Muslim",
    "preferred_categories": ["Full Length", "Accessory"],
    "price_range": [50, 200],
    "tags": ["abaya", "modest", "embroidery", "premium"],
    "past_purchases": []
}

print("User Profile:")
display(user_profile)

print("\nGenerating Recommendations...")
recs = get_recommendations(user_profile)

# Format nicely into a DataFrame
df_recs = pd.DataFrame(recs)
display(df_recs[['name', 'category', 'price', 'rating', 'score', 'explanation']])